# Backtest: Amundi EURO STOXX 50 II UCITS ETF (ISIN FR0007054358)

Yahoo-tickers: `MSE.PA` (Parijs), `MSE.MI` (Milaan), `LYSX.DE` (Xetra), `LYSX.F` (Frankfurt).

Vergelijkt **buy & hold** met een **wekelijkse switch-strategie**:
- verkopen als de koers in 5 handelsdagen ≥ **+4%** stijgt
- terugkopen als de koers in 5 handelsdagen ≤ **−4%** daalt
- liquide geld krijgt **1% rente per jaar**

Tik linksboven op **Runtime → Run all** (of het ▶️-icoon bij elke cel).

In [ ]:
!pip install -q yfinance pandas matplotlib

In [ ]:
# ---- parameters (pas gerust aan) ----
# Probeert tickers in volgorde tot er data komt (MSE.PA = Amundi-notering Parijs).
TICKERS         = ['MSE.PA', 'MSE.MI', 'LYSX.DE', 'LYSX.F']
START           = '2010-01-01'
END             = None          # None = tot vandaag
START_CAPITAL   = 10_000.0
SELL_THRESHOLD  =  0.04         # +4% in de lookback -> verkoop
BUY_THRESHOLD   = -0.04         # -4% in de lookback -> terugkoop
LOOKBACK_DAYS   = 5             # handelsdagen (~1 week)
CASH_RATE       = 0.01          # 1% per jaar op liquide geld

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

prices = None
ticker_used = None
for t in TICKERS:
    df = yf.download(t, start=START, end=END, auto_adjust=True, progress=False)
    if df is None or df.empty:
        print(f'  {t}: geen data')
        continue
    close = df['Close']
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    close = close.dropna()
    if len(close):
        prices = close
        ticker_used = t
        print(f'  {t}: {len(close)} dagen — gebruikt')
        break

assert prices is not None, 'Geen van de tickers gaf data terug'
print(f'\n{ticker_used}: {prices.index[0].date()}  →  {prices.index[-1].date()}  ({len(prices)} dagen)')
prices.tail()

In [ ]:
from dataclasses import dataclass

TRADING_DAYS_PER_YEAR = 252

@dataclass
class SimResult:
    name: str
    equity: pd.Series
    positions: pd.Series
    trades: int

    @property
    def total_return(self):
        return self.equity.iloc[-1] / self.equity.iloc[0] - 1

    @property
    def cagr(self):
        years = (self.equity.index[-1] - self.equity.index[0]).days / 365.25
        return (self.equity.iloc[-1] / self.equity.iloc[0]) ** (1 / years) - 1 if years > 0 else float('nan')

    @property
    def max_drawdown(self):
        peak = self.equity.cummax()
        return (self.equity / peak - 1).min()

def buy_and_hold(prices, start_capital=10_000.0):
    shares = start_capital / prices.iloc[0]
    return SimResult('Buy & Hold', prices * shares, pd.Series(1, index=prices.index), 1)

def weekly_switch(prices, start_capital=10_000.0,
                  sell_threshold=0.04, buy_threshold=-0.04,
                  cash_rate_annual=0.01, lookback=5):
    daily = (1 + cash_rate_annual) ** (1 / TRADING_DAYS_PER_YEAR)
    p = prices.values
    invested, shares, cash, trades = True, start_capital / p[0], 0.0, 0
    equity = np.empty(len(p)); pos = np.empty(len(p), dtype=np.int8)
    for i in range(len(p)):
        if not invested:
            cash *= daily
        if i >= lookback:
            pct = p[i] / p[i - lookback] - 1
            if invested and pct >= sell_threshold:
                cash, shares, invested = shares * p[i], 0.0, False; trades += 1
            elif not invested and pct <= buy_threshold:
                shares, cash, invested = cash / p[i], 0.0, True; trades += 1
        equity[i] = shares * p[i] + cash
        pos[i] = 1 if invested else 0
    return SimResult(f'Switch ±{int(sell_threshold*100)}% / {lookback}d',
                     pd.Series(equity, index=prices.index),
                     pd.Series(pos, index=prices.index), trades)


In [ ]:
bh = buy_and_hold(prices, START_CAPITAL)
sw = weekly_switch(prices, START_CAPITAL, SELL_THRESHOLD, BUY_THRESHOLD, CASH_RATE, LOOKBACK_DAYS)

summary = pd.DataFrame({
    'eindwaarde (€)': [bh.equity.iloc[-1], sw.equity.iloc[-1]],
    'totaal rendement': [f'{bh.total_return*100:.2f}%', f'{sw.total_return*100:.2f}%'],
    'CAGR':             [f'{bh.cagr*100:.2f}%',         f'{sw.cagr*100:.2f}%'],
    'max drawdown':     [f'{bh.max_drawdown*100:.2f}%', f'{sw.max_drawdown*100:.2f}%'],
    'trades':           [bh.trades, sw.trades],
}, index=[bh.name, sw.name])
summary

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
bh.equity.plot(ax=ax, label=bh.name)
sw.equity.plot(ax=ax, label=sw.name)
ax.fill_between(sw.positions.index, 0, sw.equity.max()*1.05,
                where=(sw.positions == 0), alpha=0.1, color='red',
                label='liquide (switch)')
ax.set_title(f'{ticker_used} — portefeuillewaarde (start € {START_CAPITAL:,.0f})')
ax.set_ylabel('EUR'); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## Parameter-scan (optioneel)
Hoe gevoelig is het resultaat voor de drempel?

In [ ]:
rows = []
for th in [0.02, 0.03, 0.04, 0.05, 0.06, 0.08]:
    r = weekly_switch(prices, START_CAPITAL, th, -th, CASH_RATE, LOOKBACK_DAYS)
    rows.append((th, r.total_return*100, r.cagr*100, r.max_drawdown*100, r.trades))
pd.DataFrame(rows, columns=['drempel', 'totaal %', 'CAGR %', 'max DD %', 'trades'])